# Simulasi Streamlit: Alur Tugas Statistika

Notebook ini membuat dan menjalankan UI Streamlit yang menjelaskan proses tugas dari **data collecting** sampai **visualisasi dan interpretasi**.

- Tidak melakukan machine learning.
- Tidak membutuhkan dataset asli untuk menampilkan simulasi.
- Angka dan grafik di aplikasi merupakan data contoh yang deterministik.
- Antarmuka dibuat minimal, netral, dan berfokus pada urutan kerja.

## Rancangan aplikasi

Aplikasi menggunakan satu halaman dengan navigasi sidebar:

1. **Gambaran umum** — menjelaskan alur lengkap.
2. **Task 1** — sumber data, unit observasi, dan struktur dataset.
3. **Task 2** — simulasi missing value, duplikat, dan cleaning.
4. **Task 3** — statistik deskriptif dan cara membacanya.
5. **Task 4** — pemilihan grafik berdasarkan pertanyaan.
6. **Ringkasan akhir** — status task dan kerangka laporan.

Interaksi disimpan dengan `st.session_state`. Setiap perubahan widget membuat Streamlit menjalankan ulang skrip dan menggambar ulang UI.

## 1. Instalasi dependensi

Streamlit resmi dijalankan sebagai server dengan perintah `streamlit run`. Di Colab, port server akan dibuka melalui proxy notebook.

In [ ]:
!pip install -q --upgrade streamlit pandas numpy
!streamlit version

## 2. Membuat aplikasi Streamlit

Kode aplikasi sudah dibundel di dalam notebook agar cukup mengunggah satu file `.ipynb`. Cell berikut mengekstraknya menjadi `/content/streamlit_app.py`.

In [ ]:
import base64
import gzip
from pathlib import Path

APP_GZIP_BASE64 = (
    "H4sIAAAAAAAACs1cbXPbRpL+rl8xBdeVySyEkJRkO4ypOzvZJN7EW46VpOpKdrGawJAcA5iBZwaWdDr/rv2+v+yqewbAACQlx9lNHT9IIjAvPT398nRPj0RZKW2Z5df2SkN1dCTcA1mX1Q0Dw2TVPKpAZmDwWZU1z4zVHMpCWHxs7NHRkbGJ4XZZwYYvUyXXYjM6YowxemCFLfgielbUmj2TUAgjDPsWLERx10ikSi7+riR3zwq4UbVdRFci476ZkMIKKJZGZHwFemksWL6I+DWSyLMoPho7SkrQeaaupCMhiiL6/dTYm4Kf099zrZRlt/Q3fo6Phczn7MH08Ww6W30dPC9ry7M5e/Do0ePp4yx8UwjJ5+xBtuZnHMIXptZrSPHd+tH6ybr3DtKUSztnD06mZ+v+eO7VsVFrfM+/Wk/Wp+79xyP6lRj7rKoCqleQ5hutaokErunTDZiqQuk5+wB6RKsb98a6zMDCseXGimwRGXvheBq9PTC8G8avzA9FbZTOuD7WYrO1czatrplRhch8e2RRf95kVag0P06VtCAk18F0JVwfX4nMbudsOp1NqutulgqyTMjNsVXVnM2S2Znm5e7blbJWlXN22r71k26nMdvOYrY9CaY7wB+SPW4t18emglTIzZwdT5LJ7GwwaMJv+Eqrq0NDuu0MRl0r3F3xP3zOJsnjJ70l0Lsr7rj4eDI5TMskmU7CnqjCx1aDNGulyzmrq4rrFAzvmpSgN0K27JkkOwxKCg7ZoYWQBuxfxzSZ9LcCN/x469cxTR5/vW93Hz/qbe6AvOke8lLQIXlO6O4Ut1A2IRO1mbPpPpHC+aaal/irv5Q7lasUslvm6ZNm4JDeY1mXq56A/zniMXnyO8SjRzCZ6U/TkL4MzD6RWLfTSOQJ8ppN9soi0ZKq6uYzBHKSfDW7SyDPzvpzSWXDFXuJKTia4JOBdO3s2K59DEx40O4OPrZSOEmeEEtQFncZRhI6YdPQ8nVOgVfHfTvkl0HW8h4d2SFgsmf0Ala8+KzdeNLfjU8wVnsE+vHeJX+mtB4W1UdnO6LqWDhJulV4EjLxYehBX3KrRdp3oJ9lp77aa6b2C8hBI3WYzr86uKT/1ZSu1PWx2UKmruZMKjkwMIjVanOcKblnzx7M1o9Wp+v79mUwlqq4/BShbLqtlbI9e/w71OSzJT6Uo5O9qMVNPxSwBy9ByJdc1jFzdMdsyyHr0f9BGLEShbA3c7YVWcZlMwT+fPplgHejyGPoWhpY8yUUhbpabm1ZLH7RNXfI+b+MTVJIt3yJInOU8TVb1aLIlhkvFT0bjdnxOauyBMH7dxpKPm+G/0ZzsJxl3HJdCimMFSkzUFYFZ9iVrZVmdsvZry+YEWVdgBVKJg0413LDFkxWiQaZqTLJ+Brqwi613IxOZ2NqgyxmCzadTegrbPCblptESMs3XJvR9EnMHp3F1HKBP1zHVSnc4GkhqhH2kEqXUIxOJslZzM6SJ2GXmE0fx+z0Kz9pqXKu/UTpVomUjy4jqaKYRTfcRG+DrjGrFpeT5PFXMdqM6Vs/Ar8edl/zEgqOQ9DvcAzXJ92KItNcDhc4idmjncaab4SS/TlaIUFatd1yMBbnoy9X3H0xqu7e0Bd689aJSsNytzR61BAHesMNW7Bumuls4nYFP3+hvfmCzU7CZ7gLX7Dp5Cx41q7zC3bqt9W9GDV8Xzg2j9kXbDQ9m0wm7UizR5Nx0CPY1knMZtPJZIdRQ/IbifBPYjalXhiEjp0aam5rLXsS362500TSAdjwaI4rj/vPDb+O5igFg+erUkRzXEtCBnw0Gw8aNMyJ5i2fhkMTk3B0+mPw1slFNPcCsjM6LZoGp7/2kOEsCdoGNAYUpqMVEnIzykWao8cwVseMfLH/O+Mm1aJC7aYnZDKQpc5UhOH5+uHTTHxgaQHGLCIfT0Xnt27sj0+/zMSH84fxAaPltcsmNPuIfo7vmQPjnOj8NqDx/lnc4i2YfInQdORw/Z6VI2a9b8md02gsH35CEnGOyFntQ699aBGd37o//Bru7kOURue39PuTeuB6ovNb/DVsP/jaupe7XEwoSojgllpdjQha/tt42WDjO/jZAdzo/JZ+38ebDn3+f+EmcmRJAor4akR/NVIqpB2wUCD7k1VtrQp4GP2CeT7BVrARIDHXxgwvuAEREJPzm8U6ShV6dcuXt8FMH4Nm9qbii6jSogR9E67FUHLQZX6WlA1YfAeFaZbj6PPbbLgxQkmX5UuaObMEsixcYecC0BIoMHa0jn4Bk7MedSwT1i2wWRUyT6xZu5osYlJZJuTO5K3kHSCJLZjhdjR245mtulqmqih4ivbl94466M4WjDhEO10qttiFZEdHr559/9cLtmCXDox9D+UKNEhWl3Xp2e9YMmX//AclXlkzg9z0GsyoATe5FpURBNx670/w/YUFi/guBzT22NSKda/ZKTb7TZgac70wGOi1kJscDEgG+VboKD56e3R0dCXslrjhEpG72h49ePCAUQoZfAo5aq19CmTLR9EFIUsjmK03lJpuKRUyq43VuO8N3ytI+SgyJRQFSgM+RheH22kTjHFUoB6vRCG2XjsCkSbWd1/JgCw7YL6IkNFQGUpPe53182fig8i4HjmCKq02mhuz/ABFjTQUXI4Oy9yYfclOd5b/yg3iVt+ttBl71J9kh33r6PaeWT+yDLRgp+QOQ1UiM6w080kvIZkGueGjaczOAr0W66DF4Xm6DkMp6L2gSXv+PQgzo3NnBRoD8M9/NPQ2Hn9nrLtsbfPp7A0vDP+jhGIMu4fQFS/q8l9E7n5xC6UGDYLhtpOXTude8kykULAX0tQaZMrZN8oELdsxXqqM+7jOCFpCnYNkWzCiYGshgVQMDaRTscXQTDlO9iBmoHxcbkBaQGPRPnzJyxWkgJpjuGmtQswM2JpZ2ELFMiigdA+uILd12P9ZVYgcqUVfV/Ky4ho2gFQDWhlnQUjcK16i9oN0kSxGtYBPZY3mDZ2l5brS3IIRCeugSPQMLR3LQLKNhrXIWSY8SdBMvgV5g+ZYWrVltMayzoHBBrRfGb4v6wy2LBMVbKEUyY4t2RW7nrRhmjM6f2qsVnJz/p3KazPH/AB9bYw4mWmJWYWe2U7YLyKDnEEGyAiV8QJkzCrNM5EbETOwUDOOFgW7lJBuheSs4KClkJtkR3zvQTT4K1XFFDFgMaOfJ/Tz1BnmVBV1Kc3o1Akh+Q1s36liB9W918PwdsfroQBJbqHCHTekexgDCMvUynD9AXBtyBBjdZ3bWhM3DLdJ1J95dnjmGc08cKc08QbqTFgnT7kySm5iltUkFBaRcMXd5DmvalujkKUFB2LoYPqTw9Of4GSdt3ZTe6WpDU5k0NfiduY1aCuKmJmcX0lujJu+4jIVBcjhpKeHJz3FaQLf72cl9+m14AbkhpVcvoMrWLGKa4tKQOol8TmqVV+pGheDLqNeuUzYKPoGNGD7zaaWpLiNAWoNlI8yop8ANXHrRMG5cvJgpN4echCzuBVQsS3qqKPlHS/A4Ni2flc7wUeljBuVzVXWblVRkz2j5VVcFzXLBHI7OUgPCcivK8BxpNWqcMuGnBAVi773C6u8/fkAWiDEoH2SoIVq5SJmdY42mswTtkCa3nEpjGf7YSpITjz+7zl2ZFYDKbg2oqxwfI7MYYYbwS7aQoASjNgyIpx264gXobE/DD/vNPyN/nZP9qNX3xhNPolTiXJRAom9t9oF3AAa4EZWkD0ZVGBZJtpNBgPFcW3qQt5Az5L/KCx4t4wCtxW2lhsGFWqS/Jrl+Lrs7AnuJs/RfUi2QrPRuqeh7Va1TvnSZe2cgUPmp3alrgM+XDjQFMBo/FxGP8JmU3Bk6wFPjeIlNxvYsm8ufmOGy0xo0WT4PA143BUzOsLvm9jLaTKNWZPHJN2no7FemNap43+Hcp+ChQYo9KVuQkr4N5LMxiLSrhqe13gy4RQoExXXquBbr6SOA1W9KjpRHgzstKlnwkmp0f2vQAvDSq55pbnhEs0KmQx8KSSho5pV3KD4MCAmOsOzZyJSmN+8LuLffy1RkGRdco1eHs2F2ADLwfKN0iIH1EcCMW36EU3vBkSr0izjFZcZlwfmJLv6nGSp4VrnmNtBLOBuo33gW4F/MkuNGklvpL+Wts6ZxdQrAaPWwtIuu3KOA9v8GmQKtJSKlzVYkAHBmFwIRRoPa7Q1OOrIS2sUxAN+9FRlQV619Sy+OinJeMal3W1AqhGkfsJPBXbLFiynKbf1KvHee4kwqVCQ7R+Ohtxyi0oO+svSadWxaLTqOFXGHhP4ceH93lE6foSf1HxYrkVBqedCGDt6BXY7QkLHid4UajWKvkhS8yEa7++frdkCU9GaQ7ZMzYdRO+DlxOvop3BnnGAcXI0G6Wb8FCA3NWwweXNjtyoMdD8h9tm/i1FdIbtdmsQmSO/SP9Kjh84Xf3Px28PYpY0epubDw/GbPYyNxJq1YwlDeRXKau1tvJdhTfdx9Blr78l4m0BrUw55oBCNhu5LeFFSeYd39yWBsJejwM1+Z/ue5vbjPvyU05iVs5iVJzEr96Nq1ywp6Ux7FD1H44kWZ5qcnDwJ1L2ctW1+VIUqsc3j8P1J+/7vzjhii9OwxWk3QmctsdVJzw52oeYvZOhWOFht0foVsGWpw85NhNYEos7M9YO7Jsbq21myD2s64MHcWoLWbvRkfHgPYzxz5UshM37dnBTQgvrJ2OkBLLQv03Y/FJr1AlgMIeIAABKkaYfMlcxEP/dG3b7x7b1nWHGN0N/BGqhq55kkgqWylptcyIR5WFxxXa8AcegWdI3eFGF9LhgUlM/LmogfQ1bXpPE7FW6XAZZj/FEMI1jIsiVFPymeIZNAWoWGG9XLCIJUHm87P96ESlHMKJUVHAjhWKUwBld450hSFCB88DUYhwZa8bXSSE0nD9PZOMEEvlcmse5T3uld27fKklTJFOzo0j2L/btEFCq9vJy8ffs2ZmIjle7LUjC+X81w9AQHmMXuIPGtO86UID1H11jrsGiaZlpVHZ1mtLMMbH7pRkqEkTAaJyBvRoGhCpuwRb/HWhSFhFHvWclBjsaNUrQmp29uThwFQ1ODIohYO4opDeoW4b1iYHK+7aRASOubJe0ys9E4MXU5Gjc9A2PU3/ugt1889Wt7U3cLq2Wzh/g3rdb9mfFUoDn2AgcrEx7AX7jFtIrqAk1DKZ3w2Y9NrB+99crRArJu7vl+m9UQ9jusVW9wWsyBsf1CP3fohjnzPag22MHoB2d8oBASc2oI+Zy7IPzeInqfMcuEj8EIwOdkU/bhZu92ur2OXpRVjeifoYSyd3SMgfapwIwlBSMCQ3AyFW4uarPCePuYfhLkDs3H/qkbfxbM3cTzpcpqE/skKdLhsiQUN6BRbi1rCbkE78aaaIJJVWJGdf+sr3zKxqWeMApFF7Ctizpm7xxuV1aVmBRiZO8prhBwg6GC3GwEWwlDwew7yATLOf6i1ANWR/mkELbshwzGJr4QX4+in8QWLKuUVTQf5kg6Se8f8+2Cxnth/z5Qm62XNAGa6/Vhc7eblVcaswl1iVlcH8ClCKVMH6I1H7Fup7p0/Q7Zy0MUNt2I0sFI3o4eHGPfON7QHuwzvpd/BwKCOwDxfqgzOwB17jg0vB/xnITQ5a6Tx34DjPQxu1WXO6nOFSdtygSuelU3ad5KGURLTq15jldcyl4K6BdKAKI9cGpreA4ZaA+cjCgoS+iycxYTQYSputRSBS6x0QAuSmtQe0qu8VVtB7DIRfXFTl7IZzDboN/nHaKYXbZlPbErSPLwIA6qidqyNC0oFEV4c9lM5TzP+ymdaWOD5H0N0oqCjybJ7Mx1fX+y9/Vj/1q8R+Tx/oQds/dTl2FSV4RG3k/ZMVZhsy+wEb2iEmDX/C+DV6q2heCaCs1Gntynbqwx+1/WPDp3Y4y9u/YnCHh+gKcHeHawP8ZJO9CBljeK2Tq69atyOjWPk9n6o7eyaYc7MNe20wEfDbt0gOPCZgnL+AfhclJdR2OzYa8uIrrwyfh+j5xfBV3cZtYl1losrReXTypYa5UlmrPL6KWQguoFWPQzZejaVUY/U77rJeSmafHi59eUHPa+huH3oGyQhieMhUM3LBJyNI7Z+ynqYo9pMXt/0j2Ea3wi3uu4lYC3e0rS7slcxnSj487UZYdweuz7PKQzSJlhlB6KEjsfLnqQAthCxZfo+tiCRa+xZFtjJIvGIkOAgsdsGHFh568D48VSTBpqREItaOC50iznLAeJRzdB1maHrKd/lKwVXMH299IltOiRtZNMOjCtOx8iOLLiOuM5Jlt6s2qOhcW4SlGuAEHK0b2n8+s7zkr9ERMdmgVHprcdhR/Zw8FoL7lFwPPi59foADKPwW4xyGgEevyx72iwGhz6uYvdg9NPOfPfnzJp9dRnS4yQN1B4WNkG5w0eFZKOPI3ogGKTR2lSBHQw3BgfvIBQLQ85q86BPgeJHj4PkuC4i2BAE/IOQiZf2Iplyfwaf/la1p4fC2uSiQbKMXYTUtxOL1Y3oz6RsY8neOb0uPObzgHiJ4HNZnQZpaqWFJ80MN0JINFms8abUoemgDagq2dmPJF35AV/D6Bui5vuRdSRt28EOS9DkPs2caWwKz4aJ78ME6lNv8uHL35+/RARa/vg8dl/PHzLjrsHM3xweIDmWJlG2SHCubToD+LPkwP4c3812v3Q8zRElv3+/ZqPsN33TaEHHdmGAh4cdCeMYtrwONxyTJQVfOU64TkNIUQyDD4f12vfjxPd6fTwcJGuXyz9vQnUyoJSwR21v4ZHxt3xebeeUkhXLLY4mQRP4do/xfQM6lkQO7gXj4PmGJwupv7BOCAN9ZW01H0dyUVAcowlZJkq/T1vvAXi8NEWtL33xNRB5KC2wNW1hFamL6rP8TSuBGR74MScVfjPgWgGjVcUOWPRgq7xYjyWdWHwoXKV39FtW69qUuW6pMoS6Qb6lB7PX774pA4V1ytncUGyK4Hn39uwef8QWKwHjF3cx5LO6JCJNDHjWXuvYiuMVRsN5cjvdGBhY7YS0iym3lA6XV/xArterqNbGuZSvGVfsulkMpknk/XH4+YpRgm9N3lEQbzoag5RJh1J43FnzluKhvD4NnqNx8FYiuDpiFn0neZ5zaVBAOvG+jim/3BA4G/Udun5W/ynBMTEUTsZVS4pvYj8hf9+h/145A448iKwOgEcweIQixEmVnOsG9IxrMyVxMNujWbLmRKSV80lFZQRngrRWQUSk0SI0lZBTuhzwQj9SSb5DuG6W4UC30agwHEYFd8JVuvhG9AQOnMPcxOrls4FB2CyaXRgC8PZ/pxd9NpKaQSfiUN8SzUVyAvaLqmk/xqzAgosWXAnKx4XU8WaS3VgqoL2vKk9wWT6Jofq37qbBy1bv54gBbzn2/DabWXMrhc+a4GV020mwzO/2d9/B+8BcS4DDVtWqQLFHpNIlBcSBrb+cBejAB97NE79CrSEgdD+Ofwd+IFPZK/LBv3Z7O0T7QphnNq7A8IV1ytOkmox4Nov8H+QrWGQ6cKJw6bEhxv7TIlR2jrwYz7PsIRz/zmGBZEAz6CDAb3qRGS+wYq5XJU+CYqqG+NeNRWwoXR/3RwjIEA3GOCj4dlsoGprmiqqnqm3zXx/aOd6FxQ8vA6LPm8DzPgxOCdqA+ku98wKqBQCXsrPErki750muVozXtRIPcXjyB2xveOw4xXIrG4daVPkGgZlTps3tcgO5+Z283P4iV61ABYzaXuT/NG3HTijw7DdvPD+fm1WgKpXXQHw4bYBmGwOog41/qHR9qyG+4j4UWHKxogmxjncfJBfpN5OHCh/+UMDueg8S11XhaKw/TlotwHsS/wfBc3jC2chWfP9Bw62hGqYxXTpRvwcKBYJ9vYP14qcugC2MVSHL0PUtq7CoHP3SlX76rm7z0dFxVvhTiuwiMNfzbA8xzrM5rTP1KbG09dGTyjUxOIDKksfFHM449AEohS6HtNPQjBYCrUTkJIhwXNKwvoOnLtrelBS6NCpwHR+V7HvDN/27i0054vDepeT+f3nRqfzYYDfDvFxcLUpZkgqBhsd3YmwvDRh7jRYaAIV1nXeq+mYcojmrLm8uHurcrCl0Zwo2fMeV1vjxe7owtdxf+q9K/KSLHoeXj+K7laInjL0TFvAg88oo+rXnF54uZRAkYwXz52Cdlda/IprLI6n87imVPYCj9fCmx6+UJZBTtX8w4sf7+oSE6VUbRSW0je2dbeY3lUf/6057gsLO35Ccn1ZOJVk+fIWD9apEoTOP7Dy2JVSGVdxZYXM0MntmY7OZJ6j/+4SgSEBCFdiHxXE2EbiYaU/grr3woe/ptTVsbt8776aZGft8+CSk8N3Xe4luFkR3i4aFJGR76o4rr6mCn5fBp7VufPMe0k4o0JoujGS4/8eoWp7N2VTbIElawZrupXBtbsyNWQslmYUIFDUV7A6hlysgK730CSa4/86XFpeVoUrS4ui6G+/fvvrT29k918MW3PCvoPcKl+8/pzrHPDGccMMV2DxzJeVsx+54VsM7t7IN3KasLa+v5MRXO8bOUvYK3e/rRGoN/IkRDSdPXsjTxN2V5ryjTxLcGZRVnWBUz/Crx3P3simMgCV2pdJL3cuiv8qs3rLcne+Dq0udmYCKV8MuNe9pQJgtFyLqBlj6cdI7LXt5R6xEZ6xfFkVIOQ9N8y7RNZ9V2cXCxZcYCJLk6Z4PxYNJiE/ugHYVpG6Ui1vDxsZ7AcUxiZCrhWNgK1cLcDghlF7jOLyNf42Ll0kdABz959X9sC++09A0Xl7y7m7L4jBKS/gHV1A+uc/HHT1hY6Nh0aDhzVV7b3MNi9AWQIMJVZK5U77+pD98L8N+j/F3uDaSlQAAA=="
)

APP_GZIP_BASE64_V3 = (
    "H4sIAAAAAAAACr1cbZPbNpL+Pr8CR9eVpSzFSJoZ21GiuYudTeJNvOdk7FRdjadUIAlJMEmQBsB52Tn/rv2+v+yqGwAJUtRobO+uVDUjEW8NoLvx9AvEi6qUmmh2o68lrY6OuHkg6qK6JVQRUblHFRUpVfCsSt0zpSWjRc41PFb66OhI6Ugxvarohq2SUqz5ZnRECCH4QHOds2XwfV5L8r2gOVdckR+opkHYVuJJKZZ/LQUzz3J6W9Z6GVzzlNlqXHDNab5SPGUxlSulqWbLgN0AiSwNwqOxoaSgMkvLa2FICIIA/3+n9G3OzvDzQpalJnf4GV6TSbxZkEezb+D9rffYjrUgj+ZTePtlFRUsh5ITeO+UTGii+RWDCgm8/QpcZAvyaD2Ht/+8qDVLF+QRfQZvvyTnAro6xle/YKK0LAXM4OQJvP1ymiRMaGg6i5+yZ7tFE1WuoXx2enx6Yif48Qj/RUp/X1XeOsU0yTayrEW6IFdUjmDdxm2XSZmX0pVwkXlF61LoyZoWPL9dkOCcbUpG3r4kf1DJaZyzIGwfBiFRVKiJYpKvO/RcpFTTiWZK83QZKH1udie4vJdEu4ceMXEpUyYnkm+2ekFm1Q1RZc5TWx9WdPyQcV+UeU4rxZ7XWpciuAwHKie2TvqiFFqWeYfWlKsqp7cLIkrByH8Y6aJCP3DSZyTlV4s1l0pPki3PU6/riqYpF5uJLqsFmUUzyYoH9nohS5BWSVNewkJWHZI3tFqQaTR/9mUd5jRmub9rdkdoymu1IMfVzbf9qcCwT+aSFWQaPW1Gh5eWVCiueSkW3vaT2XxaKMKoYl9K6GJbXjG5h8kezZ/B+8vHoGrERVXrRbJlScbS8b1c7euYDmvfTNSWpuX1gnChmIa1JFMytc2MzI8fSm2v5JXVrMDMlAsGdarwEzt5QSvYK7+Pu2EVguqwS2sU52WSTRLX2Gta0JvJNU/1dkFmx/PpAAs10nDa4R9XGpdal8WCHA+W5gzU5HywzGqSttBSu52FZDsPyfZ43xS7WjJnWjM5URVNkOMn02g69wdENXrNzHCn06663s68UbCm4n9jfZrxwNjaLmbRzDsSCio3XDTrMI2O24VyY/Rn443TKpmhufSOFnbLYlleP2Tre8NMo1b57KzJiVuTwfnMd+YT5Yymn0XEN/et6unpt0N8+fRJhy179M2iHQaKaFXFVA6dGeuceV3RnG/EhGtWqAUBEWeyLXxfK83Xtyg1CAZgU9gkZvqaMeHRw0Uzh5NnPqVWPzeU7j8y23ktyMSKGpmg3p5FJ4OCtyDTHo/uEZHuokwQVw6z4jT65ol8sOC4Dgum6RcypOtRU5VNuvzd7NxG8rTtCL5NNCuqnGo2Scq8LoRakJN5dUNma0lorctv++fv0+5C3rv7/q4++YJd9c8gPHIrKpkDK93tjJ7gtg8KnFuZ+w/VWQrvgZZcpOxm3yZ1Trd7dqkpugL8KfRE1AWTPFkQTeM6pxIeqIHR+yx3CPF+FjfiSElZ3YYkSqhM8fPn8eW9qv/kdEcV4QE5jWYtuR2i0OT6LEqePvmixRdlZ1zLuOY4P8C1/1wev3fGuwKwqxPhkCRT0PVdEe7u28nO+uMKGDvvE/hvL48pzaqefvqERfUm+tRA8oHe+xj/EyT16ckXcQsO/9miOjD2zjIOiMzO2Z3yqx38zLTkSddexVVv4NFDNsKa+wMGrTWfpg9jyXuZ/dAk/qB5zfZD94EFfsAG7odz+8n5s/H/yMFVfZhycOs27xyOwybXHlsLLPg+E1Jdq0kKpv0nCcEglN2n1nujlRUTX3pSuC7XZam7B/TDFcRnHwy+UM2jYTvNEHDIAfFalhvJlHKOEvt3CG9M9u/KQzvf68rY0YzGW0TOSIwfjPUcKf1DeS3ykqa98v09d7nVlHUmskdXHOTrewT5EMv3iDcY79AUd5HgwGR2ZOWAjbBDy0XGRboMKskLKm+dt24/Vf3693pjdqj7lCk8mj6dzWdPBhgupopds3gZKJazRDtuG/a4/Mhz9raCmTD5gyyrv5WCDTsllQbv+4+SFl0NvstgD3VKvqGxar1bmsY5V3rXcbiDeh6Ozh46cGfQru9mutdx0/OjHOJFJGvf0jx6Rbl4xUQdEqNBQ7JlsCdep1dc8ZjnXN8uyJanqTPCP+Lf7772QhVBYMMftVB0zVY0z8vr1VYX+fKNrJkJevy30lFCky1bwfIcpWxN4prn6SplRYnPRmMyOSNVGjU7v3Ddv5AMUH3KNJMFF1xpnhBFiypnBJqSdSmJ3qK/XvGizim47yIXV5FiQ5ZEVJGkIi2LKGVrWud6JcVmdDIfYx1Q9mQJHln8SjfwTYpNxIVmGybVaPYsJE9OQ6y5hD+mYVxw03mS82oELUQpC5qPjqfRaUhOo2d+k5DMnobk5Bs7aFFmTNqBkm3JEza6CEQJsYZbpoJLr2lIquXFNHr6TQhG6+zS9sBu+s3XrKAmXIH//T5MG3TESyb6E5yG5MlOZck2vBTdMRomAVql3jKqNIyHX66Z+aLKui3BL1hipd0tuZkaPnLEUblhiixJO8xsPjW7Aq8/4d58RebH/jPYha/IbHrqPWvm+RU5sdtqCkZu3ZdmmcfkKzKanU6n06an+ZPp2Gvhbes0JPPZdLqzUH3yHUfYJyGZYSuIH46NGEqmayk6HN/OuZVElAG6YcECZh52nyt2EyyAC3rP44IHC5hLhIfAaD7uVXCLEyyadep3jYsEveOHXqnhi2BhGWSnd5w0do6fBsgwmgR0AygDjLCCFuJiM8p4kgE2VlqGBK0k+zllKpEcnfP4BFUGLKlRFX5kdf34O0A9SU6VWgbWlxuc3Zm+P373dcqvzh6He5SWlS4d4egj/Ds+MAb4aYOzO4/Gw6OYyYPnYgV+lJGoi3hw5uBgOTTlFr46zQcvn0Tn2QqM5t5bBV1YwdmdIcdOY6dJ98lgRziF4OwO/w/1M9gKJhuc3cG/wbGHyel0gX6g4ExxWvWq9742p9d9J5jPqWC6r2R5PTJBsX/VVjn3xz1b1fowgrM7/L9vpzotDu6J36Lx7u3Zjy9dTViRFfI/mKEj/OSEgAvdW0IOyx8Z9NuuYfAGMkA4iemGUwFZGATQqKLcIyZjt8t1kJQAGjRb3XkjffSq6duKtZC6fY7hEQiWY3zCFowNVXZzFVOKl8JkfURupDSiaerPqz1XQL2UVOnROnhDVUY6NJGUazMtNxdYMr4mzRzSgIhSEy52Bm/4bQ9JZEkU06Ox6U9ty+sVZAGwBJTWp/baa06W5EeaK4b7W5RkuYvzjo5ef//Tn8/JklwYhPcTLWIqqSB1URd2bc2SzMg//o6JOMSNIDadCnOswFQmeaU4osFO+TGUn2uqATRmFE4QqKr5ulPtBKr9wVUNuT+019HvXGwyqqggNNtyGYRHl0dHR9dcb3E1bALOjowHjx49arOJvlc1JgEEzTGSmCjzyKQdZUy+pwbNZkyxLdVUtHUxHDYKVEHzHBgBHsORCTupI4zVe/Lwmud8a8XB42Fc9fYraoxVC/S9VBDH32YgpaOUX/GUyZEhqLK+hdUVuNjIkuRMjPaz25h8TU52Zm0dFETXG6rambq+R91BdlZtHdwdGPUjSank5ASPV1+KUO+WEnLJYiaB1SUVGzaaheTUE2m+9mrsH6dt0GeAnYOuixc891twZhSAk/1//N3R6xDETl/3KVf3alUNyxX7UkLBczdAaMzyuvgnkTvMbj7XgC5QTLf80orbc05vKaFWzoakqOnlVZkyaykqjpOoMyrIliqekzUXFIXsnkw9b3VMZNY7pncL3ZHbSS3sH6S7rSDYG5ztmxYyiZ2C15f38UGGOTfYG42hrh42/NIB5q2KGVLZWNDMMd63HX7lVvPVQtcZKRh43jNFAT/Vma4lqsQQCmImFd/CPmlW1FTAQ7HZcl2LDVFOxXsaPiQttApSKqD+ba1qAd4Np+kj8j1od/C3UMjppCDunBRM1hWFwVAlu4WOdjTjrhB1ZAeCYsHZd8bHefZjmdVqAd4T/OpoxfOmQxWOGpE3PKUZAcoqVpQpy2HWlWQpzxQPCdW0Jgz0IzQpaLLlgpGcUSm42EQ7wngAkOF2yzIBpZuUOZi4G0Fz+GzOGRv6H11Mo5M5+CFOn12G4DdbBjmYeVbS8GT0OuqipDo2zqZR8FbWwMe4/7YpvFprKJjOwHuwc/6T4Bz1D9hVXJMyVkxeUVgRWMYO5yimoz19z7HvHnQgwfd1yjURPKecZKUqxSYkaV3lPKMagH7FzDBJzigu857uj5HOBnnAt9e1gj4U4AXYyaymUnNY6IxdC6aU6bliIuE5Fft6PoG+PLACX3+SdM0zckvFBtj8Pb2mMakYOP9uKXIxAGPNZCWZpoqLW+q6N0Cm2ep9u/XCrIEwgu2R5ivWJ9N2NzoKFkNrEG11rgGvPQ5NlgQg4sVF4zy4vIxg7NGT6TiSTIB7xDLg8s7zMAQv7MePHZLAA7qCWnpkBgDDLC/lMrB5xgF4PcG5ujw+9pw9ahYSNQ+JOu7y/LFfJSowwjgK/sdNFvbgydSf1byp9Ds4hKVhr3VwZ8i5eGzpfnwZFYyK0XgRRtP1R7+L46aLVzSD1Sz29kBvvA6OBnaPSopKlCatNmtORmvRBr9S0IVbgmJnUCSCJwRSFuiaRGjNaUW2NKeF0avvWU4V6ub6PermSpYK3F5gMZVbkpWplZuM5TWeHMirFZN5TVIOdEV76UFRfRtT6AeTlA0v0wxxPAl+qgXq6gpIBk1quQ0lS1DJy0ZcQ1JncCSh/xhqAE3vmeCKbFCG9lOBEm1tzQ6mhMVyaBYOqaKC/hksDlFMcXLeXEcoqOJbgoTDQEdHLPdP4P1Gz72nsWnmn6zDNpOtvKWV0Q0F8gRFBWVPOjgCM5LyjV1SWJ6UVlSTlDebTBXNJ7Wqc1Aj/in7C9fUIkLvcKYVJVUtviUZFBdMME0r6Ad2k2UKB4lBWxNq4UP/oC1rmYA8gwPaCKYJOMXljbcO5ljwjTd4XQS/0M0mZ7Csr1jKE5qTlwKBScLIi9L4qN+KzYZuyYvzP4hiIuWSO2e1pQFCNCHBeEzvOJxFs5A4lzyqUwzn7FGk/+vzfUI1dWi2y3Xm7PsLcqY7m3BXFctqSCcwApTyikFoaWuF1KxAVcd5y8q9jo00dU5OFGqqaxJTyRWgH1ZJppiAswJ2SkEhFwjMa1IxBezToLs9A6HAmAsULIfPfy6AkUx6BWgVQTTfUJJRzTal5Blo6ZSJDRyv1pMOh+WG8vYASVnFRMrEnjHxcHyOvORWrUVRTSeawm6DfmBbDh+JxkqO0x33G1SqIYpwTTNdo8w2u2xic3u2+XcqEopTqQCutoaIc2T5LA3ZEVIr6HVkuTXwTFF3zJapFyJwL3dHKkpZyoTerYCi4bkZ/VdF9ZYsSYZDbus4sqBpBZgWIrXD3WGXW6ZByKn8ujBSNeFOqiZJqfQEkapxKg320q6H/0rU1WrNc4yiQJB29Jrq7QgIHUdyk5fxKPgqStRVMB5un67JEqIqktF0lairUdPhxdTK6ENWZxwpLXk16kVO4JVTsanpBhyFt3pb+ibNA8zu4V0MahMYR+ecjoDelX0kR4/NWfzi/I/HoXFRPk7U1ePxu4GFDfiaNH1xhd489KAOVh5cMNd8HHzG3Ds83jhrg3OLOTJPIJyEWucqXNrTyXaA8Q95HMGMMQObQe+t3xHYrqcBXsUsJMU8JMVxSIqTrqY/8as14Ow56ExQNLPo+PiZJ+VFiwF/KfMS0Fvw1C9vAd5fjU6EGid+jZO2h1ZJQq3jPQj8Deq3GDqrNSi9nG5J4rB7x5htbe6KSbpBhUeFprKoswaMuXWimq4xRAko3QDzZ+OdrQshWYCtMH7kQlw4j66bf7YH+Qx5cw8Dn3nHpQDmW+jBPQQwTZdZKVLe9e9isxe2vj0HYibBNDMghla1OYcEQKOiFpuMi4hYEFwxWccUUOeWyhrOzoLnPOOE5ugzTp1jSdxSW8WdMpX1dWSAy/O+c4Gm6QoNzwSSH5APdQlqGoRJcQRQFl2bU9tZqUFI0GfqRTKhr4IrBTO8tyff7u31gx3FbF1KoKZlg9l8HEFoyMoQX3cpb8WtaVulUVKKhOrRhXkW2rKI52VycTG9vLwMCd+IUnZ5yevfzqbfewQdzEMTAb80cXhBhV3RNaQLLl3VVJZVS6ca7UwDql+YniKuBB2NIypuR55+8quQZbfFmue5oKPOM2PpOaFoNM2gqdnXMMCCgKyDEP3tZhL2DPQ0zQ8tF3ChbbWomWY6GkeqLkZj19LTQd2991rbyWO7pjU21zReuT2Ezzhb8zFlCQctbBmOxsrPHDk3k2kE1ZiVqk5BXXnPfmFVrWtFRXBphaOBX+3Yi2FV5Qg7rKQ6feIc9nRp5/eJPbqlWAwgVm+/gp+NqqE5F+CvAThnzgTE5q1nZQuqiKTcOT+hNEMNMoSJ7dnS7mzwsqhqQPYE+JG8x8AYaKMc3OBoaHAwr1ExmLGwTgy29AT/Ipz2lcXw0O7Q8sZ2tnpRprUKrecd6IDCV0CPUcGNHi1oJpx/2FkKRJQFuOmHR31tHWjGwQcWJij8bZ3XIXlvMHmpywJccwS1O9oM6K8GY3nDScwVGqrvaQqua/iHbgVI/LMuOqjZNQeUjuxVfzkKfuVbqklV6hLHA/9Hy9fdwPEuIDwI6YcAa7pe4QCgnNf7ldtOOwiFGdUDgS6b+w54SXVxmHvxdTPUhWm3Tzvuo9A1Q0p7PVmtubePoX6sWt3bZnxw/faA/XvA7jCwme8BNveEoQ/jm2MfqNwXy+5WACsePFd1Qaqe4zlmKE0ph1nHtfOcV6UCbGTEmmWg3YqOe+cNOvdAHxixVSyjKZUWJimeowfQeN40OHkQQbVuo4oap4WDV+iywProOGNxrXsgyFjs+Y7Px3onG4Pe+hSCkLQO5NDkzVkwEHpJb032pORoZqLn2Q1lzpkPM8ySgArRh5oKzXM2guuCpumH48Hip7aYfwCc8eGYTMiHmfEeldeIPT7MyASu35KvoBIW1VVlio7Jn3pFZa1zziTmQ44sud+Zvsbk/4h7dGb6GNvD2SxdMgtJMg9JchySZNiQSVqIAZrXepfNrBqn9LxxSictygA/2k4DeNRv0sKLc51GJGVX3Pib2oZKp/1WrdlzbkMj3RYZu/aamM2sC8jZWUFqtwWZh/MqG2EJFuQieMWFdbIHv6H3rZll8Bv6sjw3fPDyt9/R8WvPGgLfvexW7B4RFXTtloiL0TgkHyDG0F20kHw4bh+CNz8EJggbDrgcyJw84JUM8YcE7nVLtsCms3yfBHB6XjCwwH0OImf9ufbM+y2t2ApOPLIkbawEdUQKuATCnGBWQeNvPZ0F14hTJgEANViBZSUElElGBcTPPEfMDlnffSlZMb2m20+li0veIWvHP7RnWBO+RhQSM5myDPwnnVElg7R3mCUvYgrY5Ohgrsf6nli1HzHyQtZ3LYUfyeNeb6+YBpzz8rffQe+nFnrdgSXh+Hj8sXu+mMB7N8g+mEVyKINk2B3SiKf1hGDkM7dosrHAHQzlAoPHirf40PlInB8AA/NO5+Dvo6z2nVHtufmcCjjYM8+vDbtIFZUIuD27yKZdQ9I8u4F/NtO6c3z5GfNIA7oN2wHROMeC+HbUJTK0ZgRLjRx78dameUQ3m9FFkJQ1pDqSwKFzw4BIm07dIYoNXHq3R1dHu1gid119nwKfmxyPg/g5sNoMAeaFD2kvI5OfHbPROHrTd4m6dhePX/72+2PAp82Dp6f/+fiSTNoHc3iwvwMX0sdedogwB1jwhWjzeA/aHM5mPAw0T3wc2W3fzR7w69nMg5Rj8NXnay//ICJowZoIq8GAmoETLGexaQQRF8SDqA+sr61Tv2sVmjhzP0yId4JW9jIPCGOO3t2W2rd+8LcNhLfzKbgwGYfL46n3lN7Yp+B6AfHyLAVT8NSrDqbocmYfjD3SQExROM3XkVh6JIeQh5iWhf3dOLiaZNAQZDEcjH0aQOylfJh0Il+5dFn1OcTVCgrL7p1dRhn8V481vcomr6tgopY1/NAe5AaCqVFmZXZPs20d1yjKdYGpOTaP5CEtnr96+aAGFZOxUbRUkGsOkeytX70bzuXr3sIuDy1Jq3RQM6qQsLS57LPlSpcbSYuR3WlPsYYk5kItZ1Y/GlmPWQ5NL9bBHXZzwS/J12Q2nU4XkMYxcU/BJuiUZAGa7LxNXAWeNCSNx60Wbyjqg+G74HcI7EJSgaUjJMGPkmU1gwTlhZ3exzH+YiJivlHTpHPMwo8cmhybZrCdNJvxYRhyDwp56WkdD4VAmocGexLyMtaOdDAis1JA2FqC2jKqBPlVMgGeRQOjfFBWUQEuIQBnsecB+lwMgh9RJd/DXPeLkHe2IRYwKwyCbxirOdgdVvDPcItuI12uzMnrYciB1Ct/C/3R/j27aKUVnQbW7wawFrMjYC1wu0Qp7NeQ5DSH5AMTNbFwGFP+jGMDHBO45y6LBBzlm4xW/9Ld3KvZupkBCYVfG3NrbbYyJDdL66OA9PvGb2EX3+3vv2LtKcBbQiXdkqrMge0xeRaTqBTFNNvKgH9rcrhD/ZpKQXtM++9Z39458MDlNb6ff/fydok2KS1G7E3wL2YyZsipGuysYYb/wmX1bUtjRexXJdbKGFIlqpTagB/1eYrFH/vfo1gACbCUtjCgk2cIi68g9y0rC+vyBNENYa9cCrHP3d+6oAEAdAV2PSiezYZWTXZShXkw9daN90U717nlYuE1GKAuKenOw4wfB/Jto9bTTHJalQB40RuL5PKsEzsyWWMsr4F6NMNNnv09oY3XVKR1c5BuLUr3jTIjzZuap/s9cbveOHgFrxsAC36zQZd+8EMLzjD0tesFHm7XOAMwD9UkX++v64FJF3baV/lnJ+1pTQ8R8UsJnhrFnY2zv3rPm4itDTugt/JnB7kwelXeVHmJ1vpzKs0GkK/h11Dc43OjIYn7/jOjuqBV32dpnIvw2pP/4e3t56Z/nBi71emnPSbpayZqXVe+rbl7E68pem4uf2JW8JabkATkZdhLPZplkEjpQnoKroGgRBvxQAsT8gkwyb+Xn2F0grM/0WKd4N/2LsrOvRDQHxCMRIhvMLm53UkLtBhazp8t7svWnUNp55aICyL2U1iOF4eDQyeLvl3fdPGxdy0uJEAq2Bgt3RH+fqPvKfUmCj9RyUR6UMDB0xAsiLvzunsFt7elwQIpGSiH2dZ4BeDcJmI/9M4eHo4keO5fXQvul4OODHQ0mrcGD0+I6uaKnlt2FBTtFsuVO4noJiX4NZOQ1I6xNpfieg6hM4zI2c2zCa6EZpiF378n874uwBuKeUN+CrzTpLtJ8CZr+C8ulOenaPwK5Np0bkyusokqFppjTgfGNiBj2CRFKZM7pblI4UgbGA7jLc/xtkLj9vMJAHACN8DgKA6hjoBApA0vHbxaQ8AbXHj558apO5RLbHQ7DGm9URbNtZ4W70aEf4Wrlw6GJ1XFYPY1Zt7b9O20zsw5PEjCKSYw1zlXJIOfv8EseTOkS6SA5DMFudilgrmbhDNYWEi7yCkHDo9pPKEZjynehsJBJIPfBlq533WFcEMQ/OXtD29/fSfai4qNFiE/0kyXNun8OZMZhfvpbjFM8oS740x+cZf93ol3YhaRJi+/5RGY7zsxj8hrvCjSMNQ7cezjl1aNvRMnEbnPKflOnEYwMi+qOoehn8DXds3eCRf1B1m26c2rnR8TeCvSeguX0vDWRyOLrXYAype91WtLMXEXFNYycH2sbB+RvnG/HACvgkMlCKR8XeWUiwO/QtC6rQ7dtl4uyUkvKz2Bu3gj0JOI8yjcu2zSQE3SlVWDjge75oPSERfrEnuAWibO37sZ1MRKjHfGXuAGmasNnNy9UNuB9ubHqIIzlyrcJqOiKcpy+h4vDv3j7wao2pRFdzCDwoN8qeYib+MFQJ8AGA5xWWZG+roAff8F2f8HqI7wmohkAAA="
)
app_code = gzip.decompress(
    base64.b64decode(APP_GZIP_BASE64_V3)
).decode('utf-8')
app_code = app_code.replace('use_container_width=True', 'width=\"stretch\"')
app_code = app_code.replace('use_container_width=False', 'width=\"content\"')

app_path = Path('/content/streamlit_app.py')
app_path.write_text(app_code, encoding='utf-8', newline='\n')

config_dir = Path('/content/.streamlit')
config_dir.mkdir(parents=True, exist_ok=True)
config_path = config_dir / 'config.toml'
config_path.write_text(
    '''[theme]
base = "dark"
primaryColor = "#31B7E8"
backgroundColor = "#191919"
secondaryBackgroundColor = "#202020"
textColor = "#F2F2F2"
font = "sans-serif"
''',
    encoding='utf-8',
)

print('Aplikasi dibuat:', app_path)
print('Tema dibuat:', config_path)
print('Ukuran:', f'{app_path.stat().st_size:,}', 'byte')
print('Jumlah baris:', len(app_code.splitlines()))

## 3. Validasi kode

Cell ini memeriksa sintaks sebelum server dijalankan.

In [ ]:
!python -m py_compile /content/streamlit_app.py
print('Sintaks aplikasi valid.')

## 4. Menjalankan server Streamlit

Server berjalan pada port `8501`. Log disimpan di `/content/streamlit.log`.

In [ ]:
import subprocess
import sys
import time
import urllib.request

log_file = open('/content/streamlit.log', 'w', encoding='utf-8')
server_process = subprocess.Popen(
    [
        sys.executable, '-m', 'streamlit', 'run',
        '/content/streamlit_app.py',
        '--server.port=8501',
        '--server.headless=true',
        '--browser.gatherUsageStats=false',
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

server_ready = False
for _ in range(30):
    try:
        urllib.request.urlopen('http://localhost:8501', timeout=1)
        server_ready = True
        break
    except Exception:
        time.sleep(1)

print('PID server:', server_process.pid)
print('Status:', 'siap' if server_ready else 'belum merespons; periksa log')

## 5. Membuka preview dari Colab

Tautan hanya aktif selama runtime Colab dan server Streamlit masih berjalan.

In [ ]:
from google.colab.output import eval_js
from IPython.display import display, Markdown

preview_url = eval_js('google.colab.kernel.proxyPort(8501)')
display(Markdown(f'### [Buka simulasi Streamlit]({preview_url})'))
print('Jika halaman belum tampil, tunggu beberapa detik lalu buka tautan kembali.')

## 6. Pemeriksaan dan unduhan

Gunakan cell berikut jika aplikasi tidak terbuka atau jika ingin mengunduh skrip Python-nya.

In [ ]:
print(Path('/content/streamlit.log').read_text(encoding='utf-8')[-4000:])

In [ ]:
from google.colab import files
files.download('/content/streamlit_app.py')

## 7. Menghentikan server

Jalankan cell ini setelah selesai menggunakan simulasi.

In [ ]:
if 'server_process' in globals() and server_process.poll() is None:
    server_process.terminate()
    server_process.wait(timeout=10)
    print('Server Streamlit dihentikan.')
else:
    print('Tidak ada server aktif.')